In [ ]:
# Jalankan cell ini dulu di Colab:
!pip install requests beautifulsoup4 pandas lxml

In [ ]:
# ============================================================
# SCRAPER DETIK - KEYWORD BANYAK + TAHUN SPESIFIK
# ============================================================

import re
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup

# ============================================================
# CONFIG
# ============================================================

TARGET_YEAR = 2020
MAX_PAGES = 20
OUTPUT_FILE = "detik_2020_pages.csv"

KEYWORDS = [

    # ===============================
    # Regulasi & Hukum
    # ===============================
    "perlindungan data pribadi",
    "UU PDP",
    "RUU PDP",
    "undang-undang perlindungan data",
    "regulasi data pribadi",
    "hukum data pribadi",
    "aturan data pribadi",
    "kebijakan data pribadi",

    # ===============================
    # Kebocoran & Serangan Siber
    # ===============================
    "kasus kebocoran data",
    "kebocoran data",
    "data pribadi bocor",
    "data bocor",
    "hacker data",
    "serangan siber data",
    "peretasan data",
    "serangan siber Indonesia",
    "cyber attack Indonesia",
    "data breach Indonesia",

    # ===============================
    # Lembaga & Aktor
    # ===============================
    "BSSN kebocoran",
    "Kominfo data pribadi",
    "PDNS diretas",
    "Kominfo kebocoran data",
    "BSSN serangan siber",
    "Kemenkominfo data bocor",

    # ===============================
    # Kasus Spesifik Indonesia
    # ===============================
    "kebocoran data BPJS",
    "kebocoran data KTP",
    "kebocoran data e-KTP",
    "kebocoran data Dukcapil",
    "ransomware Indonesia",
    "data bocor Tokopedia",
    "data bocor Bukalapak",
    "data bocor PLN",
    "data bocor IndiHome",
    "data bocor Telkom",

    # ===============================
    # Tambahan Penelitian
    # ===============================
    "keamanan data pribadi",
    "privasi data Indonesia",
    "perlindungan privasi",
    "keamanan siber Indonesia",
    "insiden kebocoran data Indonesia",
]

session = requests.Session()

# ============================================================
# HEADER
# ============================================================

def headers():
    return {
        "User-Agent": "Mozilla/5.0",
        "Accept-Language": "id-ID,id;q=0.9"
    }

def sleep():
    time.sleep(random.uniform(0.2,0.5))


# ============================================================
# PARSE YEAR
# ============================================================

def parse_year(text):

    if not text:
        return None

    m = re.search(r'(\d{4})', text)

    if m:
        return int(m.group(1))

    return None


# ============================================================
# REQUEST
# ============================================================

def get(url):

    try:
        r = session.get(url,headers=headers(),timeout=10)

        if r.status_code == 200:
            return BeautifulSoup(r.text,"lxml")

    except:
        return None

    return None


# ============================================================
# BODY
# ============================================================

def get_body(url):

    soup = get(url+"?single=1")

    if not soup:
        return ""

    body = soup.select_one(".detail__body-text")

    if not body:
        return ""

    return body.text.strip()


# ============================================================
# SEARCH URL
# ============================================================

def search_url(keyword,page):

    return f"https://www.detik.com/search/searchall?query={keyword}&page={page}"


# ============================================================
# SCRAPE
# ============================================================

def scrape_keyword(keyword):

    print("\n")
    print("="*70)
    print("🔍 Keyword:",keyword)
    print("="*70)

    data=[]

    for page in range(1,MAX_PAGES+1):

        print(f"\n📄 Scraping halaman {page}")

        soup = get(search_url(keyword,page))

        if not soup:
            break

        items = soup.select("article.list-content__item")

        print("📰 Total ditemukan:",len(items))

        for i,item in enumerate(items):

            try:

                title = item.select_one("h3 a").text.strip()
                link = item.select_one("h3 a")["href"]

                date_raw = item.select_one(".media__date").text.strip()

                year = parse_year(date_raw)

                # hanya ambil tahun target
                if year != TARGET_YEAR:
                    continue

                print(f"✅ [{i+1}] {title[:60]}")

                content = get_body(link)

                data.append({
                    "source":"Detik",
                    "keyword":keyword,
                    "title":title,
                    "date":date_raw,
                    "year":year,
                    "url":link,
                    "content":content
                })

                sleep()

            except:
                continue

    print(f"\n🎯 Total {TARGET_YEAR}:",len(data))

    return data


# ============================================================
# MAIN
# ============================================================

def run():

    print("\n")
    print("="*70)
    print("🚀 SCRAPER DETIK DIMULAI")
    print("Target Tahun:",TARGET_YEAR)
    print("Total Keyword:",len(KEYWORDS))
    print("="*70)

    all_data=[]

    for i,kw in enumerate(KEYWORDS):

        print(f"\nProgress Keyword {i+1}/{len(KEYWORDS)}")

        d = scrape_keyword(kw)
        all_data.extend(d)

        print("📊 Total sementara:",len(all_data))


    df = pd.DataFrame(all_data)

    print("\n🧹 Menghapus duplikasi...")
    df = df.drop_duplicates("url")

    print("📊 Total Final:",len(df))

    print("\n💾 Menyimpan CSV...")
    df.to_csv(OUTPUT_FILE,index=False,encoding="utf-8-sig")

    print("\n")
    print("="*70)
    print("✅ SCRAPING SELESAI")
    print("="*70)

    print("Total Artikel:",len(df))
    print("File:",OUTPUT_FILE)

    return df


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    run()



🚀 SCRAPER DETIK DIMULAI
Target Tahun: 2020
Total Keyword: 39

Progress Keyword 1/39


🔍 Keyword: perlindungan data pribadi

📄 Scraping halaman 1
📰 Total ditemukan: 12

📄 Scraping halaman 2
📰 Total ditemukan: 12

📄 Scraping halaman 3
📰 Total ditemukan: 12

📄 Scraping halaman 4
📰 Total ditemukan: 11

📄 Scraping halaman 5
📰 Total ditemukan: 11

📄 Scraping halaman 6
📰 Total ditemukan: 11

📄 Scraping halaman 7
📰 Total ditemukan: 11

📄 Scraping halaman 8
📰 Total ditemukan: 11

📄 Scraping halaman 9
📰 Total ditemukan: 11

📄 Scraping halaman 10
📰 Total ditemukan: 11

📄 Scraping halaman 11
📰 Total ditemukan: 11

📄 Scraping halaman 12
📰 Total ditemukan: 11

📄 Scraping halaman 13
📰 Total ditemukan: 11

📄 Scraping halaman 14
📰 Total ditemukan: 11

📄 Scraping halaman 15
📰 Total ditemukan: 11

📄 Scraping halaman 16
📰 Total ditemukan: 11

📄 Scraping halaman 17
📰 Total ditemukan: 11

📄 Scraping halaman 18
📰 Total ditemukan: 11

📄 Scraping halaman 19
📰 Total ditemukan: 11

📄 Scraping halaman 20
📰 Tota

In [ ]:
# ============================================================
# SCRAPER DETIK - KEYWORD BANYAK + TAHUN SPESIFIK
# ============================================================

import re
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup

# ============================================================
# CONFIG
# ============================================================

TARGET_YEAR = 2021
MAX_PAGES = 20
OUTPUT_FILE = "detik_2020_pages.csv"

KEYWORDS = [

    # ===============================
    # Regulasi & Hukum
    # ===============================
    "perlindungan data pribadi",
    "data pribadi"
    "UU PDP",
    "RUU PDP",
    "undang-undang perlindungan data",
    "UU No. 27 Tahun 2022",
    "regulasi data pribadi",
    "hukum data pribadi",
    "aturan data pribadi",
    "kebijakan data pribadi",

    # ===============================
    # Kebocoran & Serangan Siber
    # ===============================
    "kasus kebocoran data",
    "kebocoran data",
    "kebocoran data pribadi",
    "data pribadi bocor",
    "data bocor",
    "hacker data",
    "serangan siber data",
    "peretasan data",
    "serangan siber Indonesia",
    "cyber attack Indonesia",
    "data breach Indonesia",

    # ===============================
    # Lembaga & Aktor
    # ===============================
    "BSSN kebocoran",
    "Kominfo data pribadi",
    "PDNS diretas",
    "Kominfo kebocoran data",
    "BSSN serangan siber",
    "Kemenkominfo data bocor",

    # ===============================
    # Kasus Spesifik Indonesia
    # ===============================
    "kebocoran data BPJS",
    "kebocoran data KTP",
    "kebocoran data e-KTP",
    "kebocoran data Dukcapil",
    "ransomware Indonesia",
    "data bocor Tokopedia",
    "data bocor Bukalapak",
    "data bocor PLN",
    "data bocor IndiHome",
    "data bocor Telkom",

    # ===============================
    # Tambahan Penelitian
    # ===============================
    "keamanan data pribadi",
    "privasi data Indonesia",
    "perlindungan privasi",
    "keamanan siber Indonesia",
    "insiden kebocoran data Indonesia",
]

session = requests.Session()

# ============================================================
# HEADER
# ============================================================

def headers():
    return {
        "User-Agent": "Mozilla/5.0",
        "Accept-Language": "id-ID,id;q=0.9"
    }

def sleep():
    time.sleep(random.uniform(0.2,0.5))


# ============================================================
# PARSE YEAR
# ============================================================

def parse_year(text):

    if not text:
        return None

    m = re.search(r'(\d{4})', text)

    if m:
        return int(m.group(1))

    return None


# ============================================================
# REQUEST
# ============================================================

def get(url):

    try:
        r = session.get(url,headers=headers(),timeout=10)

        if r.status_code == 200:
            return BeautifulSoup(r.text,"lxml")

    except:
        return None

    return None


# ============================================================
# BODY
# ============================================================

def get_body(url):

    soup = get(url+"?single=1")

    if not soup:
        return ""

    body = soup.select_one(".detail__body-text")

    if not body:
        return ""

    return body.text.strip()


# ============================================================
# SEARCH URL
# ============================================================

def search_url(keyword,page):

    return f"https://www.detik.com/search/searchall?query={keyword}&page={page}"


# ============================================================
# SCRAPE
# ============================================================

def scrape_keyword(keyword):

    print("\n")
    print("="*70)
    print("🔍 Keyword:",keyword)
    print("="*70)

    data=[]

    for page in range(1,MAX_PAGES+1):

        print(f"\n📄 Scraping halaman {page}")

        soup = get(search_url(keyword,page))

        if not soup:
            break

        items = soup.select("article.list-content__item")

        print("📰 Total ditemukan:",len(items))

        for i,item in enumerate(items):

            try:

                title = item.select_one("h3 a").text.strip()
                link = item.select_one("h3 a")["href"]

                date_raw = item.select_one(".media__date").text.strip()

                year = parse_year(date_raw)

                # hanya ambil tahun target
                if year != TARGET_YEAR:
                    continue

                print(f"✅ [{i+1}] {title[:60]}")

                content = get_body(link)

                data.append({
                    "source":"Detik",
                    "keyword":keyword,
                    "title":title,
                    "date":date_raw,
                    "year":year,
                    "url":link,
                    "content":content
                })

                sleep()

            except:
                continue

    print(f"\n🎯 Total {TARGET_YEAR}:",len(data))

    return data


# ============================================================
# MAIN
# ============================================================

def run():

    print("\n")
    print("="*70)
    print("🚀 SCRAPER DETIK DIMULAI")
    print("Target Tahun:",TARGET_YEAR)
    print("Total Keyword:",len(KEYWORDS))
    print("="*70)

    all_data=[]

    for i,kw in enumerate(KEYWORDS):

        print(f"\nProgress Keyword {i+1}/{len(KEYWORDS)}")

        d = scrape_keyword(kw)
        all_data.extend(d)

        print("📊 Total sementara:",len(all_data))


    df = pd.DataFrame(all_data)

    print("\n🧹 Menghapus duplikasi...")
    df = df.drop_duplicates("url")

    print("📊 Total Final:",len(df))

    print("\n💾 Menyimpan CSV...")
    df.to_csv(OUTPUT_FILE,index=False,encoding="utf-8-sig")

    print("\n")
    print("="*70)
    print("✅ SCRAPING SELESAI")
    print("="*70)

    print("Total Artikel:",len(df))
    print("File:",OUTPUT_FILE)

    return df


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    run()



🚀 SCRAPER DETIK DIMULAI
Target Tahun: 2021
Total Keyword: 41

Progress Keyword 1/41


🔍 Keyword: perlindungan data pribadi

📄 Scraping halaman 1
📰 Total ditemukan: 12

📄 Scraping halaman 2
📰 Total ditemukan: 12

📄 Scraping halaman 3
📰 Total ditemukan: 12

📄 Scraping halaman 4
📰 Total ditemukan: 11

📄 Scraping halaman 5
📰 Total ditemukan: 11

📄 Scraping halaman 6
📰 Total ditemukan: 11

📄 Scraping halaman 7
📰 Total ditemukan: 11

📄 Scraping halaman 8
📰 Total ditemukan: 11

📄 Scraping halaman 9
📰 Total ditemukan: 11

📄 Scraping halaman 10
📰 Total ditemukan: 11

📄 Scraping halaman 11
📰 Total ditemukan: 11

📄 Scraping halaman 12
📰 Total ditemukan: 11

📄 Scraping halaman 13
📰 Total ditemukan: 11

📄 Scraping halaman 14
📰 Total ditemukan: 11

📄 Scraping halaman 15
📰 Total ditemukan: 11

📄 Scraping halaman 16
📰 Total ditemukan: 11

📄 Scraping halaman 17
📰 Total ditemukan: 11

📄 Scraping halaman 18
📰 Total ditemukan: 11

📄 Scraping halaman 19
📰 Total ditemukan: 11

📄 Scraping halaman 20
📰 Tota

In [ ]:
# ============================================================
# SCRAPER DETIK - KEYWORD BANYAK + TAHUN SPESIFIK
# ============================================================

import re
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup

# ============================================================
# CONFIG
# ============================================================

TARGET_YEAR = 2022
MAX_PAGES = 20
OUTPUT_FILE = "detik_2022_20p.csv"

# KEYWORDS = [
#     "perlindungan data pribadi",
#     "UU PDP",
#     "RUU PDP",
#     "kebocoran data",
#     "data pribadi bocor",
#     "kasus kebocoran data pribadi",
# ]

KEYWORDS = [

    # ===============================
    # Regulasi & Hukum
    # ===============================
    "perlindungan data pribadi",
    "data pribadi"
    "UU PDP",
    "RUU PDP",
    "undang-undang perlindungan data",
    "UU No. 27 Tahun 2022",
    "regulasi data pribadi",
    "hukum data pribadi",
    "aturan data pribadi",
    "kebijakan data pribadi",

    # ===============================
    # Kebocoran & Serangan Siber
    # ===============================
    "kasus kebocoran data",
    "kebocoran data",
    "kebocoran data pribadi",
    "data pribadi bocor",
    "data bocor",
    "hacker data",
    "serangan siber data",
    "peretasan data",
    "serangan siber Indonesia",
    "cyber attack Indonesia",
    "data breach Indonesia",

    # ===============================
    # Lembaga & Aktor
    # ===============================
    "BSSN kebocoran",
    "Kominfo data pribadi",
    "PDNS diretas",
    "Kominfo kebocoran data",
    "BSSN serangan siber",
    "Kemenkominfo data bocor",

    # ===============================
    # Kasus Spesifik Indonesia
    # ===============================
    "kebocoran data BPJS",
    "kebocoran data KTP",
    "kebocoran data e-KTP",
    "kebocoran data Dukcapil",
    "ransomware Indonesia",
    "data bocor Tokopedia",
    "data bocor Bukalapak",
    "data bocor PLN",
    "data bocor IndiHome",
    "data bocor Telkom",

    # ===============================
    # Tambahan Penelitian
    # ===============================
    "keamanan data pribadi",
    "privasi data Indonesia",
    "perlindungan privasi",
    "keamanan siber Indonesia",
    "insiden kebocoran data Indonesia",
]

session = requests.Session()

# ============================================================
# HEADER
# ============================================================

def headers():
    return {
        "User-Agent": "Mozilla/5.0",
        "Accept-Language": "id-ID,id;q=0.9"
    }

def sleep():
    time.sleep(random.uniform(0.2,0.5))


# ============================================================
# PARSE YEAR
# ============================================================

def parse_year(text):

    if not text:
        return None

    m = re.search(r'(\d{4})', text)

    if m:
        return int(m.group(1))

    return None


# ============================================================
# REQUEST
# ============================================================

def get(url):

    try:
        r = session.get(url,headers=headers(),timeout=10)

        if r.status_code == 200:
            return BeautifulSoup(r.text,"lxml")

    except:
        return None

    return None


# ============================================================
# BODY
# ============================================================

def get_body(url):

    soup = get(url+"?single=1")

    if not soup:
        return ""

    body = soup.select_one(".detail__body-text")

    if not body:
        return ""

    return body.text.strip()


# ============================================================
# SEARCH URL
# ============================================================

def search_url(keyword,page):

    return f"https://www.detik.com/search/searchall?query={keyword}&page={page}"


# ============================================================
# SCRAPE
# ============================================================

def scrape_keyword(keyword):

    print("\n")
    print("="*70)
    print("🔍 Keyword:",keyword)
    print("="*70)

    data=[]

    for page in range(1,MAX_PAGES+1):

        print(f"\n📄 Scraping halaman {page}")

        soup = get(search_url(keyword,page))

        if not soup:
            break

        items = soup.select("article.list-content__item")

        print("📰 Total ditemukan:",len(items))

        for i,item in enumerate(items):

            try:

                title = item.select_one("h3 a").text.strip()
                link = item.select_one("h3 a")["href"]

                date_raw = item.select_one(".media__date").text.strip()

                year = parse_year(date_raw)

                # hanya ambil tahun target
                if year != TARGET_YEAR:
                    continue

                print(f"✅ [{i+1}] {title[:60]}")

                content = get_body(link)

                data.append({
                    "source":"Detik",
                    "keyword":keyword,
                    "title":title,
                    "date":date_raw,
                    "year":year,
                    "url":link,
                    "content":content
                })

                sleep()

            except:
                continue

    print(f"\n🎯 Total {TARGET_YEAR}:",len(data))

    return data


# ============================================================
# MAIN
# ============================================================

def run():

    print("\n")
    print("="*70)
    print("🚀 SCRAPER DETIK DIMULAI")
    print("Target Tahun:",TARGET_YEAR)
    print("Total Keyword:",len(KEYWORDS))
    print("="*70)

    all_data=[]

    for i,kw in enumerate(KEYWORDS):

        print(f"\nProgress Keyword {i+1}/{len(KEYWORDS)}")

        d = scrape_keyword(kw)
        all_data.extend(d)

        print("📊 Total sementara:",len(all_data))


    df = pd.DataFrame(all_data)

    print("\n🧹 Menghapus duplikasi...")
    df = df.drop_duplicates("url")

    print("📊 Total Final:",len(df))

    print("\n💾 Menyimpan CSV...")
    df.to_csv(OUTPUT_FILE,index=False,encoding="utf-8-sig")

    print("\n")
    print("="*70)
    print("✅ SCRAPING SELESAI")
    print("="*70)

    print("Total Artikel:",len(df))
    print("File:",OUTPUT_FILE)

    return df


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    run()



🚀 SCRAPER DETIK DIMULAI
Target Tahun: 2022
Total Keyword: 41

Progress Keyword 1/41


🔍 Keyword: perlindungan data pribadi

📄 Scraping halaman 1
📰 Total ditemukan: 12

📄 Scraping halaman 2
📰 Total ditemukan: 12

📄 Scraping halaman 3
📰 Total ditemukan: 12

📄 Scraping halaman 4
📰 Total ditemukan: 11

📄 Scraping halaman 5
📰 Total ditemukan: 11

📄 Scraping halaman 6
📰 Total ditemukan: 11

📄 Scraping halaman 7
📰 Total ditemukan: 11

📄 Scraping halaman 8
📰 Total ditemukan: 11

📄 Scraping halaman 9
📰 Total ditemukan: 11

📄 Scraping halaman 10
📰 Total ditemukan: 11

📄 Scraping halaman 11
📰 Total ditemukan: 11

📄 Scraping halaman 12
📰 Total ditemukan: 11

📄 Scraping halaman 13
📰 Total ditemukan: 11

📄 Scraping halaman 14
📰 Total ditemukan: 11

📄 Scraping halaman 15
📰 Total ditemukan: 11

📄 Scraping halaman 16
📰 Total ditemukan: 11

📄 Scraping halaman 17
📰 Total ditemukan: 11

📄 Scraping halaman 18
📰 Total ditemukan: 11

📄 Scraping halaman 19
📰 Total ditemukan: 11

📄 Scraping halaman 20
📰 Tota

In [ ]:
# ============================================================
# SCRAPER DETIK - KEYWORD BANYAK + TAHUN SPESIFIK
# ============================================================

import re
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup

# ============================================================
# CONFIG
# ============================================================

TARGET_YEAR = 2023
MAX_PAGES = 20
OUTPUT_FILE = "detik_2023_20p.csv"

# KEYWORDS = [
#     "perlindungan data pribadi",
#     "UU PDP",
#     "RUU PDP",
#     "kebocoran data",
#     "data pribadi bocor",
#     "kasus kebocoran data pribadi",
# ]

KEYWORDS = [

    # ===============================
    # Regulasi & Hukum
    # ===============================
    "perlindungan data pribadi",
    "data pribadi"
    "UU PDP",
    "RUU PDP",
    "undang-undang perlindungan data",
    "UU No. 27 Tahun 2022",
    "regulasi data pribadi",
    "hukum data pribadi",
    "aturan data pribadi",
    "kebijakan data pribadi",

    # ===============================
    # Kebocoran & Serangan Siber
    # ===============================
    "kasus kebocoran data",
    "kebocoran data",
    "kebocoran data pribadi",
    "data pribadi bocor",
    "data bocor",
    "hacker data",
    "serangan siber data",
    "peretasan data",
    "serangan siber Indonesia",
    "cyber attack Indonesia",
    "data breach Indonesia",

    # ===============================
    # Lembaga & Aktor
    # ===============================
    "Kominfo data pribadi",
    "Kominfo kebocoran data",
    "Kemenkominfo data bocor",

    # ===============================
    # Kasus Spesifik Indonesia
    # ===============================
    "kebocoran data BPJS",
    "kebocoran data e-KTP",
    "kebocoran data Dukcapil",
    "ransomware Indonesia",

    # ===============================
    # Tambahan Penelitian
    # ===============================
    "keamanan data pribadi",
    "privasi data Indonesia",
    "perlindungan privasi",
    "keamanan siber Indonesia",
    "insiden kebocoran data Indonesia",
]

session = requests.Session()

# ============================================================
# HEADER
# ============================================================

def headers():
    return {
        "User-Agent": "Mozilla/5.0",
        "Accept-Language": "id-ID,id;q=0.9"
    }

def sleep():
    time.sleep(random.uniform(0.2,0.5))


# ============================================================
# PARSE YEAR
# ============================================================

def parse_year(text):

    if not text:
        return None

    m = re.search(r'(\d{4})', text)

    if m:
        return int(m.group(1))

    return None


# ============================================================
# REQUEST
# ============================================================

def get(url):

    try:
        r = session.get(url,headers=headers(),timeout=10)

        if r.status_code == 200:
            return BeautifulSoup(r.text,"lxml")

    except:
        return None

    return None


# ============================================================
# BODY
# ============================================================

def get_body(url):

    soup = get(url+"?single=1")

    if not soup:
        return ""

    body = soup.select_one(".detail__body-text")

    if not body:
        return ""

    return body.text.strip()


# ============================================================
# SEARCH URL
# ============================================================

def search_url(keyword,page):

    return f"https://www.detik.com/search/searchall?query={keyword}&page={page}"


# ============================================================
# SCRAPE
# ============================================================

def scrape_keyword(keyword):

    print("\n")
    print("="*70)
    print("🔍 Keyword:",keyword)
    print("="*70)

    data=[]

    for page in range(1,MAX_PAGES+1):

        print(f"\n📄 Scraping halaman {page}")

        soup = get(search_url(keyword,page))

        if not soup:
            break

        items = soup.select("article.list-content__item")

        print("📰 Total ditemukan:",len(items))

        for i,item in enumerate(items):

            try:

                title = item.select_one("h3 a").text.strip()
                link = item.select_one("h3 a")["href"]

                date_raw = item.select_one(".media__date").text.strip()

                year = parse_year(date_raw)

                # hanya ambil tahun target
                if year != TARGET_YEAR:
                    continue

                print(f"✅ [{i+1}] {title[:60]}")

                content = get_body(link)

                data.append({
                    "source":"Detik",
                    "keyword":keyword,
                    "title":title,
                    "date":date_raw,
                    "year":year,
                    "url":link,
                    "content":content
                })

                sleep()

            except:
                continue

    print(f"\n🎯 Total {TARGET_YEAR}:",len(data))

    return data


# ============================================================
# MAIN
# ============================================================

def run():

    print("\n")
    print("="*70)
    print("🚀 SCRAPER DETIK DIMULAI")
    print("Target Tahun:",TARGET_YEAR)
    print("Total Keyword:",len(KEYWORDS))
    print("="*70)

    all_data=[]

    for i,kw in enumerate(KEYWORDS):

        print(f"\nProgress Keyword {i+1}/{len(KEYWORDS)}")

        d = scrape_keyword(kw)
        all_data.extend(d)

        print("📊 Total sementara:",len(all_data))


    df = pd.DataFrame(all_data)

    print("\n🧹 Menghapus duplikasi...")
    df = df.drop_duplicates("url")

    print("📊 Total Final:",len(df))

    print("\n💾 Menyimpan CSV...")
    df.to_csv(OUTPUT_FILE,index=False,encoding="utf-8-sig")

    print("\n")
    print("="*70)
    print("✅ SCRAPING SELESAI")
    print("="*70)

    print("Total Artikel:",len(df))
    print("File:",OUTPUT_FILE)

    return df


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    run()



🚀 SCRAPER DETIK DIMULAI
Target Tahun: 2023
Total Keyword: 32

Progress Keyword 1/32


🔍 Keyword: perlindungan data pribadi

📄 Scraping halaman 1
📰 Total ditemukan: 12
✅ [8] Edukasi Perlindungan Data Pribadi dan Krisis Kebocoran Data

📄 Scraping halaman 2
📰 Total ditemukan: 12

📄 Scraping halaman 3
📰 Total ditemukan: 12

📄 Scraping halaman 4
📰 Total ditemukan: 11

📄 Scraping halaman 5
📰 Total ditemukan: 11

📄 Scraping halaman 6
📰 Total ditemukan: 11

📄 Scraping halaman 7
📰 Total ditemukan: 11

📄 Scraping halaman 8
📰 Total ditemukan: 11

📄 Scraping halaman 9
📰 Total ditemukan: 11

📄 Scraping halaman 10
📰 Total ditemukan: 11

📄 Scraping halaman 11
📰 Total ditemukan: 11

📄 Scraping halaman 12
📰 Total ditemukan: 11

📄 Scraping halaman 13
📰 Total ditemukan: 11

📄 Scraping halaman 14
📰 Total ditemukan: 11

📄 Scraping halaman 15
📰 Total ditemukan: 11

📄 Scraping halaman 16
📰 Total ditemukan: 11

📄 Scraping halaman 17
📰 Total ditemukan: 11

📄 Scraping halaman 18
📰 Total ditemukan: 11

📄 Scrap

In [ ]:
# ============================================================
# SCRAPER DETIK - KEYWORD BANYAK + TAHUN SPESIFIK
# ============================================================

import re
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup

# ============================================================
# CONFIG
# ============================================================

TARGET_YEAR = 2024
MAX_PAGES = 30
OUTPUT_FILE = "detik_2024_30p_.csv"

# KEYWORDS = [
#     "perlindungan data pribadi",
#     "UU PDP",
#     "RUU PDP",
#     "kebocoran data",
#     "data pribadi bocor",
#     "kasus kebocoran data pribadi",
# ]

KEYWORDS = [

    # ===============================
    # Regulasi & Hukum
    # ===============================
    "perlindungan data pribadi",
    "data pribadi"
    "UU PDP",
    "RUU PDP",
    "undang-undang perlindungan data",
    "UU No. 27 Tahun 2022",
    "regulasi data pribadi",
    "hukum data pribadi",
    "aturan data pribadi",
    "kebijakan data pribadi",

    # ===============================
    # Kebocoran & Serangan Siber
    # ===============================
    "kasus kebocoran data",
    "kebocoran data",
    "kebocoran data pribadi",
    "data pribadi bocor",
    "data bocor",
    "hacker data",
    "serangan siber data",
    "peretasan data",
    "serangan siber Indonesia",
    "cyber attack Indonesia",
    "data breach Indonesia",

    # ===============================
    # Lembaga & Aktor
    # ===============================
    "Kominfo data pribadi",
    "Kominfo kebocoran data",
    "Kemenkominfo data bocor",

    # ===============================
    # Kasus Spesifik Indonesia
    # ===============================
    "kebocoran data BPJS",
    "kebocoran data e-KTP",
    "kebocoran data Dukcapil",
    "ransomware Indonesia",

    # ===============================
    # Tambahan Penelitian
    # ===============================
    "keamanan data pribadi",
    "privasi data Indonesia",
    "perlindungan privasi",
    "keamanan siber Indonesia",
    "insiden kebocoran data Indonesia",
]

session = requests.Session()

# ============================================================
# HEADER
# ============================================================

def headers():
    return {
        "User-Agent": "Mozilla/5.0",
        "Accept-Language": "id-ID,id;q=0.9"
    }

def sleep():
    time.sleep(random.uniform(0.2,0.5))


# ============================================================
# PARSE YEAR
# ============================================================

def parse_year(text):

    if not text:
        return None

    m = re.search(r'(\d{4})', text)

    if m:
        return int(m.group(1))

    return None


# ============================================================
# REQUEST
# ============================================================

def get(url):

    try:
        r = session.get(url,headers=headers(),timeout=10)

        if r.status_code == 200:
            return BeautifulSoup(r.text,"lxml")

    except:
        return None

    return None


# ============================================================
# BODY
# ============================================================

def get_body(url):

    soup = get(url+"?single=1")

    if not soup:
        return ""

    body = soup.select_one(".detail__body-text")

    if not body:
        return ""

    return body.text.strip()


# ============================================================
# SEARCH URL
# ============================================================

def search_url(keyword,page):

    return f"https://www.detik.com/search/searchall?query={keyword}&page={page}"


# ============================================================
# SCRAPE
# ============================================================

def scrape_keyword(keyword):

    print("\n")
    print("="*70)
    print("🔍 Keyword:",keyword)
    print("="*70)

    data=[]

    for page in range(1,MAX_PAGES+1):

        print(f"\n📄 Scraping halaman {page}")

        soup = get(search_url(keyword,page))

        if not soup:
            break

        items = soup.select("article.list-content__item")

        print("📰 Total ditemukan:",len(items))

        for i,item in enumerate(items):

            try:

                title = item.select_one("h3 a").text.strip()
                link = item.select_one("h3 a")["href"]

                date_raw = item.select_one(".media__date").text.strip()

                year = parse_year(date_raw)

                # hanya ambil tahun target
                if year != TARGET_YEAR:
                    continue

                print(f"✅ [{i+1}] {title[:60]}")

                content = get_body(link)

                data.append({
                    "source":"Detik",
                    "keyword":keyword,
                    "title":title,
                    "date":date_raw,
                    "year":year,
                    "url":link,
                    "content":content
                })

                sleep()

            except:
                continue

    print(f"\n🎯 Total {TARGET_YEAR}:",len(data))

    return data


# ============================================================
# MAIN
# ============================================================

def run():

    print("\n")
    print("="*70)
    print("🚀 SCRAPER DETIK DIMULAI")
    print("Target Tahun:",TARGET_YEAR)
    print("Total Keyword:",len(KEYWORDS))
    print("="*70)

    all_data=[]

    for i,kw in enumerate(KEYWORDS):

        print(f"\nProgress Keyword {i+1}/{len(KEYWORDS)}")

        d = scrape_keyword(kw)
        all_data.extend(d)

        print("📊 Total sementara:",len(all_data))


    df = pd.DataFrame(all_data)

    print("\n🧹 Menghapus duplikasi...")
    df = df.drop_duplicates("url")

    print("📊 Total Final:",len(df))

    print("\n💾 Menyimpan CSV...")
    df.to_csv(OUTPUT_FILE,index=False,encoding="utf-8-sig")

    print("\n")
    print("="*70)
    print("✅ SCRAPING SELESAI")
    print("="*70)

    print("Total Artikel:",len(df))
    print("File:",OUTPUT_FILE)

    return df


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    run()



🚀 SCRAPER DETIK DIMULAI
Target Tahun: 2024
Total Keyword: 32

Progress Keyword 1/32


🔍 Keyword: perlindungan data pribadi

📄 Scraping halaman 1
📰 Total ditemukan: 12

📄 Scraping halaman 2
📰 Total ditemukan: 12

📄 Scraping halaman 3
📰 Total ditemukan: 12

📄 Scraping halaman 4
📰 Total ditemukan: 11

📄 Scraping halaman 5
📰 Total ditemukan: 11

📄 Scraping halaman 6
📰 Total ditemukan: 11

📄 Scraping halaman 7
📰 Total ditemukan: 11

📄 Scraping halaman 8
📰 Total ditemukan: 11

📄 Scraping halaman 9
📰 Total ditemukan: 11

📄 Scraping halaman 10
📰 Total ditemukan: 11

📄 Scraping halaman 11
📰 Total ditemukan: 11

📄 Scraping halaman 12
📰 Total ditemukan: 11

📄 Scraping halaman 13
📰 Total ditemukan: 11

📄 Scraping halaman 14
📰 Total ditemukan: 11

📄 Scraping halaman 15
📰 Total ditemukan: 11

📄 Scraping halaman 16
📰 Total ditemukan: 11

📄 Scraping halaman 17
📰 Total ditemukan: 11

📄 Scraping halaman 18
📰 Total ditemukan: 11

📄 Scraping halaman 19
📰 Total ditemukan: 11

📄 Scraping halaman 20
📰 Tota

In [ ]:
# ============================================================
# SCRAPER DETIK - KEYWORD BANYAK + TAHUN SPESIFIK
# ============================================================

import re
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup

# ============================================================
# CONFIG
# ============================================================

TARGET_YEAR = 2025
MAX_PAGES = 30
OUTPUT_FILE = "detik_2025_20p.csv"

# KEYWORDS = [
#     "perlindungan data pribadi",
#     "UU PDP",
#     "RUU PDP",
#     "kebocoran data",
#     "data pribadi bocor",
#     "kasus kebocoran data pribadi",
# ]

KEYWORDS = [

    # ===============================
    # Regulasi & Hukum
    # ===============================
    "perlindungan data pribadi",
    "data pribadi"
    "UU PDP",
    "RUU PDP",
    "undang-undang perlindungan data",
    "UU No. 27 Tahun 2022",
    "regulasi data pribadi",
    "hukum data pribadi",
    "aturan data pribadi",
    "kebijakan data pribadi",

    # ===============================
    # Kebocoran & Serangan Siber
    # ===============================
    "kasus kebocoran data",
    "kebocoran data",
    "kebocoran data pribadi",
    "data pribadi bocor",
    "data bocor",
    "hacker data",
    "serangan siber data",
    "peretasan data",
    "serangan siber Indonesia",
    "cyber attack Indonesia",
    "data breach Indonesia",

    # ===============================
    # Lembaga & Aktor
    # ===============================
    "Kominfo data pribadi",
    "Kominfo kebocoran data",
    "Kemenkominfo data bocor",

    # ===============================
    # Kasus Spesifik Indonesia
    # ===============================
    "kebocoran data BPJS",
    "kebocoran data e-KTP",
    "kebocoran data Dukcapil",
    "ransomware Indonesia",

    # ===============================
    # Tambahan Penelitian
    # ===============================
    "keamanan data pribadi",
    "privasi data Indonesia",
    "perlindungan privasi",
    "keamanan siber Indonesia",
    "insiden kebocoran data Indonesia",
]

session = requests.Session()

# ============================================================
# HEADER
# ============================================================

def headers():
    return {
        "User-Agent": "Mozilla/5.0",
        "Accept-Language": "id-ID,id;q=0.9"
    }

def sleep():
    time.sleep(random.uniform(0.2,0.5))


# ============================================================
# PARSE YEAR
# ============================================================

def parse_year(text):

    if not text:
        return None

    m = re.search(r'(\d{4})', text)

    if m:
        return int(m.group(1))

    return None


# ============================================================
# REQUEST
# ============================================================

def get(url):

    try:
        r = session.get(url,headers=headers(),timeout=10)

        if r.status_code == 200:
            return BeautifulSoup(r.text,"lxml")

    except:
        return None

    return None


# ============================================================
# BODY
# ============================================================

def get_body(url):

    soup = get(url+"?single=1")

    if not soup:
        return ""

    body = soup.select_one(".detail__body-text")

    if not body:
        return ""

    return body.text.strip()


# ============================================================
# SEARCH URL
# ============================================================

def search_url(keyword,page):

    return f"https://www.detik.com/search/searchall?query={keyword}&page={page}"


# ============================================================
# SCRAPE
# ============================================================

def scrape_keyword(keyword):

    print("\n")
    print("="*70)
    print("🔍 Keyword:",keyword)
    print("="*70)

    data=[]

    for page in range(1,MAX_PAGES+1):

        print(f"\n📄 Scraping halaman {page}")

        soup = get(search_url(keyword,page))

        if not soup:
            break

        items = soup.select("article.list-content__item")

        print("📰 Total ditemukan:",len(items))

        for i,item in enumerate(items):

            try:

                title = item.select_one("h3 a").text.strip()
                link = item.select_one("h3 a")["href"]

                date_raw = item.select_one(".media__date").text.strip()

                year = parse_year(date_raw)

                # hanya ambil tahun target
                if year != TARGET_YEAR:
                    continue

                print(f"✅ [{i+1}] {title[:60]}")

                content = get_body(link)

                data.append({
                    "source":"Detik",
                    "keyword":keyword,
                    "title":title,
                    "date":date_raw,
                    "year":year,
                    "url":link,
                    "content":content
                })

                sleep()

            except:
                continue

    print(f"\n🎯 Total {TARGET_YEAR}:",len(data))

    return data


# ============================================================
# MAIN
# ============================================================

def run():

    print("\n")
    print("="*70)
    print("🚀 SCRAPER DETIK DIMULAI")
    print("Target Tahun:",TARGET_YEAR)
    print("Total Keyword:",len(KEYWORDS))
    print("="*70)

    all_data=[]

    for i,kw in enumerate(KEYWORDS):

        print(f"\nProgress Keyword {i+1}/{len(KEYWORDS)}")

        d = scrape_keyword(kw)
        all_data.extend(d)

        print("📊 Total sementara:",len(all_data))


    df = pd.DataFrame(all_data)

    print("\n🧹 Menghapus duplikasi...")
    df = df.drop_duplicates("url")

    print("📊 Total Final:",len(df))

    print("\n💾 Menyimpan CSV...")
    df.to_csv(OUTPUT_FILE,index=False,encoding="utf-8-sig")

    print("\n")
    print("="*70)
    print("✅ SCRAPING SELESAI")
    print("="*70)

    print("Total Artikel:",len(df))
    print("File:",OUTPUT_FILE)

    return df


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    run()

Streaming output truncated to the last 5000 lines.
📄 Scraping halaman 2
📰 Total ditemukan: 10

📄 Scraping halaman 3
📰 Total ditemukan: 10

📄 Scraping halaman 4
📰 Total ditemukan: 10

📄 Scraping halaman 5
📰 Total ditemukan: 10

📄 Scraping halaman 6
📰 Total ditemukan: 10

📄 Scraping halaman 7
📰 Total ditemukan: 10

📄 Scraping halaman 8
📰 Total ditemukan: 10

📄 Scraping halaman 9
📰 Total ditemukan: 10

📄 Scraping halaman 10
📰 Total ditemukan: 10

📄 Scraping halaman 11
📰 Total ditemukan: 10

📄 Scraping halaman 12
📰 Total ditemukan: 10
✅ [3] Uni Eropa Tegur Israel yang Larang Organisasi Bantuan Kemanu
✅ [4] Di Lahan Kecil Bekasi, Pupuk Menjadi Kunci Ketahanan Pangan 
✅ [5] Mendagri Berikan Bantuan untuk Korban Bencana Aceh di Kecama
✅ [6] Local Hero Bandung: Simon Sanjaya Sang Pemanen Air Langit
✅ [7] Komdigi Terima Ratusan Aduan soal Perlindungan Data Pribadi
✅ [8] Mengenal Sosok Eyang Hasan Maolani yang Diajukan Jadi Pahlaw
✅ [9] Pemerintah Ingin Batasi Medsos bagi Anak, Pakar IPB: Kuatka

In [ ]:
# ============================================================
# SCRAPER DETIK - KEYWORD BANYAK + TAHUN SPESIFIK
# ============================================================

import re
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup

# ============================================================
# CONFIG
# ============================================================

TARGET_YEAR = 2025
MAX_PAGES = 30
OUTPUT_FILE = "detik_2025_20p.csv"

# KEYWORDS = [
#     "perlindungan data pribadi",
#     "UU PDP",
#     "RUU PDP",
#     "kebocoran data",
#     "data pribadi bocor",
#     "kasus kebocoran data pribadi",
# ]

KEYWORDS = [

    # ===============================
    # Regulasi & Hukum
    # ===============================
    "perlindungan data pribadi",
    "data pribadi"
    "UU PDP",
    "RUU PDP",
    "undang-undang perlindungan data",
    "UU No. 27 Tahun 2022",
    "regulasi data pribadi",
    "hukum data pribadi",
    "aturan data pribadi",
    "kebijakan data pribadi",

    # ===============================
    # Kebocoran & Serangan Siber
    # ===============================
    "kasus kebocoran data",
    "kebocoran data",
    "kebocoran data pribadi",
    "data pribadi bocor",
    "data bocor",
    "hacker data",
    "serangan siber data",
    "peretasan data",
    "serangan siber Indonesia",
    "cyber attack Indonesia",
    "data breach Indonesia",

    # ===============================
    # Lembaga & Aktor
    # ===============================
    "Kominfo data pribadi",
    "Kominfo kebocoran data",
    "Kemenkominfo data bocor",

    # ===============================
    # Kasus Spesifik Indonesia
    # ===============================
    "kebocoran data BPJS",
    "kebocoran data e-KTP",
    "kebocoran data Dukcapil",
    "ransomware Indonesia",

    # ===============================
    # Tambahan Penelitian
    # ===============================
    "keamanan data pribadi",
    "privasi data Indonesia",
    "perlindungan privasi",
    "keamanan siber Indonesia",
    "insiden kebocoran data Indonesia",
]

session = requests.Session()

# ============================================================
# HEADER
# ============================================================

def headers():
    return {
        "User-Agent": "Mozilla/5.0",
        "Accept-Language": "id-ID,id;q=0.9"
    }

def sleep():
    time.sleep(random.uniform(0.2,0.5))


# ============================================================
# PARSE YEAR
# ============================================================

def parse_year(text):

    if not text:
        return None

    m = re.search(r'(\d{4})', text)

    if m:
        return int(m.group(1))

    return None


# ============================================================
# REQUEST
# ============================================================

def get(url):

    try:
        r = session.get(url,headers=headers(),timeout=10)

        if r.status_code == 200:
            return BeautifulSoup(r.text,"lxml")

    except:
        return None

    return None


# ============================================================
# BODY
# ============================================================

def get_body(url):

    soup = get(url+"?single=1")

    if not soup:
        return ""

    body = soup.select_one(".detail__body-text")

    if not body:
        return ""

    paragraphs = body.find_all("p")

    text = []

    for p in paragraphs:

        content = p.text.strip()

        if content and "ADVERTISEMENT" not in content:
            text.append(content)

    return " ".join(text)


# ============================================================
# SEARCH URL
# ============================================================

def search_url(keyword,page):

    return f"https://www.detik.com/search/searchall?query={keyword}&page={page}"


# ============================================================
# SCRAPE
# ============================================================

def scrape_keyword(keyword):

    print("\n")
    print("="*70)
    print("🔍 Keyword:",keyword)
    print("="*70)

    data=[]

    for page in range(1,MAX_PAGES+1):

        print(f"\n📄 Scraping halaman {page}")

        soup = get(search_url(keyword,page))

        if not soup:
            break

        items = soup.select("article.list-content__item")

        print("📰 Total ditemukan:",len(items))

        for i,item in enumerate(items):

            try:

                title = item.select_one("h3 a").text.strip()
                link = item.select_one("h3 a")["href"]

                date_raw = item.select_one(".media__date").text.strip()

                year = parse_year(date_raw)

                # hanya ambil tahun target
                if year != TARGET_YEAR:
                    continue

                print(f"✅ [{i+1}] {title[:60]}")

                content = get_body(link)

                data.append({
                    "source":"Detik",
                    "keyword":keyword,
                    "title":title,
                    "date":date_raw,
                    "year":year,
                    "url":link,
                    "content":content
                })

                sleep()

            except:
                continue

    print(f"\n🎯 Total {TARGET_YEAR}:",len(data))

    return data


# ============================================================
# MAIN
# ============================================================

def run():

    print("\n")
    print("="*70)
    print("🚀 SCRAPER DETIK DIMULAI")
    print("Target Tahun:",TARGET_YEAR)
    print("Total Keyword:",len(KEYWORDS))
    print("="*70)

    all_data=[]

    for i,kw in enumerate(KEYWORDS):

        print(f"\nProgress Keyword {i+1}/{len(KEYWORDS)}")

        d = scrape_keyword(kw)
        all_data.extend(d)

        print("📊 Total sementara:",len(all_data))


    df = pd.DataFrame(all_data)

    print("\n🧹 Menghapus duplikasi...")
    df = df.drop_duplicates("url")

    print("📊 Total Final:",len(df))

    print("\n💾 Menyimpan CSV...")
    df.to_csv(OUTPUT_FILE,index=False,encoding="utf-8-sig")

    print("\n")
    print("="*70)
    print("✅ SCRAPING SELESAI")
    print("="*70)

    print("Total Artikel:",len(df))
    print("File:",OUTPUT_FILE)

    return df


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    run()

In [ ]:
def get_body(url):

    soup = get(url+"?single=1")

    if not soup:
        return ""

    body = soup.select_one(".detail__body-text")

    if not body:
        return ""

    paragraphs = body.find_all("p")

    text = []

    for p in paragraphs:
        content = p.text.strip()
        if content:
            text.append(content)

    return " ".join(text)

BAWAH DARI CLAUDE, bener tapi lama. dan beberapa ada yang benar tapi kurang tepat

In [ ]:
# ============================================================
# scraper_detik_fast.py — Versi CEPAT --- TAHUN SALAH
# ============================================================

import re
import csv
import time
import random
import logging
import requests
import pandas as pd
from bs4 import BeautifulSoup
from typing import Optional

# ── Logging ───────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ── Konstanta ─────────────────────────────────────────────
KEYWORDS = [
    "perlindungan data pribadi",
    "UU PDP",
    "kebocoran data",
    "data pribadi bocor",
]

YEAR_START  = 2021
YEAR_END    = 2022
OUTPUT_FILE = "detik_pdp_2021.csv"
MAX_PAGES   = 1   # lebih cepat

# Session (lebih cepat)
session = requests.Session()

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)",
    "Mozilla/5.0 (X11; Linux x86_64)",
]

# ── HTTP Helper ───────────────────────────────────────────
def random_headers():
    return {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept-Language": "id-ID,id;q=0.9",
    }

def polite_sleep():
    time.sleep(random.uniform(0.2, 0.6))  # lebih cepat

def safe_get(url: str) -> Optional[BeautifulSoup]:
    try:
        resp = session.get(url, headers=random_headers(), timeout=8)
        if resp.status_code == 200:
            return BeautifulSoup(resp.text, "lxml")
    except:
        return None
    return None

# ── Clean Text ────────────────────────────────────────────
_NOISE = re.compile(
    r"(Baca Juga|ADVERTISEMENT|Simak Video|Lihat Juga)",
    re.IGNORECASE,
)

def clean_text(raw: str):
    text = re.sub(r"<[^>]+>", " ", raw)
    text = _NOISE.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()

# ── Date Parser ───────────────────────────────────────────
_MONTHS_ID = {
    "januari": 1, "februari": 2, "maret": 3,
    "april": 4, "mei": 5, "juni": 6,
    "juli": 7, "agustus": 8,
    "september": 9, "oktober": 10,
    "november": 11, "desember": 12,
}

def parse_date(raw):
    if not raw:
        return None

    m = re.search(r"(\d{1,2})\s+([A-Za-z]+)\s+(\d{4})", raw)
    if m:
        month = _MONTHS_ID.get(m.group(2).lower())
        if month:
            return f"{m.group(3)}-{month:02d}-{int(m.group(1)):02d}"

    return None

# ── Extract Body ──────────────────────────────────────────
def extract_body(url):

    soup = safe_get(url + "?single=1")
    if not soup:
        return ""

    body = soup.select_one("div.detail__body-text")
    if not body:
        return ""

    return clean_text(body.get_text())

# ── Search URL ────────────────────────────────────────────
def build_search_url(keyword, page):
    return (
        "https://www.detik.com/search/searchall"
        f"?query={keyword}"
        f"&siteid=2"
        f"&sortby=time"
        f"&fromdate=01/01/{YEAR_START}"
        f"&todate=31/12/{YEAR_END}"
        f"&page={page}"
    )

# ── Scrape Keyword ────────────────────────────────────────
def scrape_keyword(keyword):

    records = []

    for page in range(1, MAX_PAGES + 1):

        url = build_search_url(keyword, page)
        soup = safe_get(url)

        if not soup:
            break

        items = soup.select("article")

        for item in items:

            try:

                title_tag = item.select_one("h3 a")
                if not title_tag:
                    continue

                title = title_tag.text.strip()
                link = title_tag["href"]

                date_tag = item.select_one(".media__date")
                raw_date = date_tag.text.strip() if date_tag else ""

                polite_sleep()

                content = extract_body(link)

                records.append({
                    "source": "Detik",
                    "keyword": keyword,
                    "title": title,
                    "date": raw_date,
                    "url": link,
                    "content": content,
                })

            except:
                continue

        polite_sleep()

    return records

# ── Main ────────────────────────────────────────────────
def run():

    all_records = []

    for kw in KEYWORDS:

        log.info(f"Scraping: {kw}")
        records = scrape_keyword(kw)
        all_records.extend(records)

    df = pd.DataFrame(all_records)

    df.drop_duplicates("url", inplace=True)

    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print("Selesai")
    print("Total:", len(df))

    return df


if __name__ == "__main__":
    run()

Selesai
Total: 285


In [ ]:
# ============================================================
# scraper_detik_fast.py — Versi CEPAT
# ============================================================

import re
import csv
import time
import random
import logging
import requests
import pandas as pd
from bs4 import BeautifulSoup
from typing import Optional

# ── Logging ───────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ── Konstanta ─────────────────────────────────────────────
KEYWORDS = [
    "perlindungan data pribadi",
    "UU PDP",
    "kebocoran data",
    "data pribadi",
]

YEAR_START  = 2021
YEAR_END    = 2022
OUTPUT_FILE = "detik_pdp_2021.csv"
MAX_PAGES   = 1   # lebih cepat

# Session (lebih cepat)
session = requests.Session()

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)",
    "Mozilla/5.0 (X11; Linux x86_64)",
]

# ── HTTP Helper ───────────────────────────────────────────
def random_headers():
    return {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept-Language": "id-ID,id;q=0.9",
    }

def polite_sleep():
    time.sleep(random.uniform(0.2, 0.6))  # lebih cepat

def safe_get(url: str) -> Optional[BeautifulSoup]:
    try:
        resp = session.get(url, headers=random_headers(), timeout=8)
        if resp.status_code == 200:
            return BeautifulSoup(resp.text, "lxml")
    except:
        return None
    return None

# ── Clean Text ────────────────────────────────────────────
_NOISE = re.compile(
    r"(Baca Juga|ADVERTISEMENT|Simak Video|Lihat Juga)",
    re.IGNORECASE,
)

def clean_text(raw: str):
    text = re.sub(r"<[^>]+>", " ", raw)
    text = _NOISE.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()

# ── Date Parser ───────────────────────────────────────────
_MONTHS_ID = {
    "januari": 1, "februari": 2, "maret": 3,
    "april": 4, "mei": 5, "juni": 6,
    "juli": 7, "agustus": 8,
    "september": 9, "oktober": 10,
    "november": 11, "desember": 12,
}

def parse_date(raw):
    if not raw:
        return None

    m = re.search(r"(\d{1,2})\s+([A-Za-z]+)\s+(\d{4})", raw)
    if m:
        month = _MONTHS_ID.get(m.group(2).lower())
        if month:
            return f"{m.group(3)}-{month:02d}-{int(m.group(1)):02d}"

    return None

# ── Extract Body ──────────────────────────────────────────
def extract_body(url):

    soup = safe_get(url + "?single=1")
    if not soup:
        return ""

    body = soup.select_one("div.detail__body-text")
    if not body:
        return ""

    return clean_text(body.get_text())

# ── Search URL ────────────────────────────────────────────
def build_search_url(keyword, page):
    return (
        "https://www.detik.com/search/searchall"
        f"?query={keyword}"
        f"&siteid=2"
        f"&sortby=time"
        f"&fromdate=01/01/{YEAR_START}"
        f"&todate=31/12/{YEAR_END}"
        f"&page={page}"
    )

# ── Scrape Keyword ────────────────────────────────────────
def scrape_keyword(keyword):

    records = []

    for page in range(1, MAX_PAGES + 1):

        url = build_search_url(keyword, page)
        soup = safe_get(url)

        if not soup:
            break

        items = soup.select("article")

        for item in items:

            try:

                title_tag = item.select_one("h3 a")
                if not title_tag:
                    continue

                title = title_tag.text.strip()
                link = title_tag["href"]

                date_tag = item.select_one(".media__date")
                raw_date = date_tag.text.strip() if date_tag else ""

                polite_sleep()

                content = extract_body(link)

                records.append({
                    "source": "Detik",
                    "keyword": keyword,
                    "title": title,
                    "date": raw_date,
                    "url": link,
                    "content": content,
                })

            except:
                continue

        polite_sleep()

    return records

# ── Main ────────────────────────────────────────────────
def run():

    all_records = []

    for kw in KEYWORDS:

        log.info(f"Scraping: {kw}")
        records = scrape_keyword(kw)
        all_records.extend(records)

    df = pd.DataFrame(all_records)

    df.drop_duplicates("url", inplace=True)

    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print("Selesai")
    print("Total:", len(df))

    return df


if __name__ == "__main__":
    run()

Selesai
Total: 46


terlalu deep scrape

In [ ]:
# ============================================================
# scraper_detik_fast.py — Versi CEPAT + Filter Tahun Akurat
# Google Colab Ready | Standalone | Parallel Requests
# Output: detik_pdp.csv
# ============================================================
# !pip install requests beautifulsoup4 pandas lxml

import re
import csv
import time
import random
import logging
import requests
import pandas as pd
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional, Tuple

# ── 1. Logging ───────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ── 2. Konstanta — UBAH DI SINI ──────────────────────────────
KEYWORDS = [
    # Regulasi & hukum
    "perlindungan data pribadi",
    "UU PDP",
    "RUU PDP",
    "undang-undang perlindungan data",
    "regulasi data pribadi",
    "hukum data pribadi",
    # Kebocoran & serangan
    "kebocoran data",
    "data pribadi bocor",
    "data bocor",
    "hacker data",
    "serangan siber data",
    "peretasan data",
    # Lembaga & aktor
    "BSSN kebocoran",
    "Kominfo data pribadi",
    "PDNS diretas",
    # Kasus spesifik
    "kebocoran data BPJS",
    "kebocoran data KTP",
    "kebocoran data e-KTP",
    "kebocoran data Dukcapil",
    "ransomware Indonesia",
]

YEAR_START   = 2020   # ← ganti sesuai kebutuhan
YEAR_END     = 2020   # ← ganti sesuai kebutuhan
OUTPUT_FILE  = "detik_pdp2020tok.csv"
MAX_PAGES    = 10      # halaman per keyword (5 × ~10 artikel = ~50 kandidat/keyword)
MAX_WORKERS  = 6      # thread paralel untuk deep-scrape artikel
SLEEP_SEARCH = (0.5, 1.2)   # jeda antar request halaman pencarian
SLEEP_ARTICLE = (0.3, 0.8)  # jeda antar artikel (lebih pendek karena paralel)

# ── 3. HTTP Session (shared, thread-safe) ────────────────────
session = requests.Session()
adapter = requests.adapters.HTTPAdapter(
    pool_connections=10,
    pool_maxsize=20,
    max_retries=requests.adapters.Retry(total=2, backoff_factor=0.3),
)
session.mount("https://", adapter)
session.mount("http://", adapter)

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 "
    "(KHTML, like Gecko) Version/17.4 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:125.0) "
    "Gecko/20100101 Firefox/125.0",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 17_4 like Mac OS X) "
    "AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4 Mobile/15E148 Safari/604.1",
]

def random_headers() -> dict:
    return {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Referer": "https://www.google.com/",
    }

def safe_get(url: str, timeout: int = 10) -> Optional[BeautifulSoup]:
    try:
        resp = session.get(url, headers=random_headers(), timeout=timeout)
        if resp.status_code == 200:
            return BeautifulSoup(resp.text, "lxml")
        log.warning(f"HTTP {resp.status_code} → {url}")
    except requests.RequestException as exc:
        log.debug(f"Request gagal ({exc.__class__.__name__}): {url}")
    return None

# ── 4. Parsing Tanggal dari Meta Tag ─────────────────────────
def parse_meta_date(soup: BeautifulSoup) -> Optional[str]:
    """
    Sumber tanggal paling akurat untuk Detik:
    <meta name="publishdate" content="2024/03/15 10:30:00">
    """
    # Prioritas 1: publishdate (paling konsisten di Detik)
    meta = soup.find("meta", attrs={"name": "publishdate"})
    if meta and meta.get("content"):
        m = re.match(r"(\d{4})/(\d{2})/(\d{2})", meta["content"].strip())
        if m:
            return f"{m.group(1)}-{m.group(2)}-{m.group(3)}"

    # Prioritas 2: article:published_time (Open Graph)
    meta2 = soup.find("meta", attrs={"property": "article:published_time"})
    if meta2 and meta2.get("content"):
        m = re.match(r"(\d{4})-(\d{2})-(\d{2})", meta2["content"].strip())
        if m:
            return f"{m.group(1)}-{m.group(2)}-{m.group(3)}"

    # Prioritas 3: <time itemprop="datePublished">
    time_tag = soup.find("time", attrs={"itemprop": "datePublished"})
    if time_tag and time_tag.get("datetime"):
        m = re.match(r"(\d{4})-(\d{2})-(\d{2})", time_tag["datetime"])
        if m:
            return f"{m.group(1)}-{m.group(2)}-{m.group(3)}"

    return None

def year_in_range(date_str: Optional[str]) -> bool:
    if not date_str:
        return False
    try:
        return YEAR_START <= int(date_str[:4]) <= YEAR_END
    except (ValueError, IndexError):
        return False

# ── 5. Pembersihan Teks ───────────────────────────────────────
_NOISE = re.compile(
    r"(Baca\s+(Juga|juga)|BACA JUGA|Simak(\s+juga|\s+Video)?|"
    r"Advertisement|ADVERTISEMENT|Lihat Juga|Tonton Video|"
    r"Editor\s*:\s*[\w\s]+|Reporter\s*:\s*[\w\s]+|"
    r"ADVERTISEMENT SCROLL TO CONTINUE WITH CONTENT)",
    re.IGNORECASE,
)

def clean_text(raw: str) -> str:
    text = re.sub(r"<[^>]+>", " ", raw)
    text = _NOISE.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()

# ── 6. Deep Scrape Satu Artikel (dijalankan paralel) ─────────
def extract_article(url: str) -> Tuple[Optional[str], str]:
    """
    Ambil (tanggal_akurat, konten_bersih) dari satu URL artikel.
    ?single=1 memaksa artikel multi-halaman Detik jadi satu halaman.
    Didesain thread-safe: tidak ada state global yang dimodifikasi.
    """
    time.sleep(random.uniform(*SLEEP_ARTICLE))   # jeda ringan per thread
    fetch_url = url.rstrip("/") + "?single=1"
    soup = safe_get(fetch_url)
    if not soup:
        return None, ""

    date_str = parse_meta_date(soup)

    body = (
        soup.select_one("div.detail__body-text")
        or soup.select_one("div.itp_bodycontent")
        or soup.find("article")
    )
    if not body:
        return date_str, ""

    for tag in body.find_all(["script", "style", "figure", "aside", "noscript"]):
        tag.decompose()

    return date_str, clean_text(body.get_text(separator=" "))

# ── 7. Build Search URL ───────────────────────────────────────
def build_search_url(keyword: str, page: int) -> str:
    return (
        "https://www.detik.com/search/searchall"
        f"?query={requests.utils.quote(keyword)}"
        f"&siteid=2&sortby=time&sorttime=0"
        f"&fromdate=01/01/{YEAR_START}&todate=31/12/{YEAR_END}"
        f"&page={page}"
    )

# ── 8. Kumpulkan Link dari Halaman Pencarian ──────────────────
def collect_links(keyword: str) -> list[Tuple[str, str]]:
    """
    Kumpulkan (title, url) dari semua halaman pencarian untuk satu keyword.
    Ini cepat karena hanya mengambil listing, belum ke artikel.
    """
    candidates = []
    for page in range(1, MAX_PAGES + 1):
        url = build_search_url(keyword, page)
        soup = safe_get(url)
        if not soup:
            break

        items = soup.select("article.list-content__item")
        if not items:
            break

        for item in items:
            tag = item.select_one("h3.media__title a")
            if tag and tag.get("href"):
                candidates.append((tag.get_text(strip=True), tag["href"]))

        log.info(f"  [{keyword}] Halaman {page}: {len(items)} link dikumpulkan.")
        time.sleep(random.uniform(*SLEEP_SEARCH))

    return candidates

# ── 9. Scrape per Keyword (paralel deep-scrape) ───────────────
def scrape_keyword(keyword: str) -> list[dict]:
    log.info(f"\n[Detik] ▶ Keyword: '{keyword}'")

    # Tahap 1: kumpulkan semua link dari halaman pencarian (cepat)
    candidates = collect_links(keyword)
    log.info(f"  Total kandidat link: {len(candidates)}")

    if not candidates:
        return []

    # Tahap 2: deep-scrape semua artikel secara PARALEL
    records = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_meta = {
            executor.submit(extract_article, url): (title, url)
            for title, url in candidates
        }

        for future in as_completed(future_to_meta):
            title, url = future_to_meta[future]
            try:
                date_str, content = future.result()

                if not date_str:
                    continue

                year = int(date_str[:4])

                if year > YEAR_END:
                    log.debug(f"  Skip terlalu baru ({date_str}): {title[:50]}")
                    continue
                if year < YEAR_START:
                    log.debug(f"  Skip terlalu lama ({date_str}): {title[:50]}")
                    continue
                if not content:
                    continue

                records.append({
                    "source":  "Detik.com",
                    "keyword": keyword,
                    "title":   title,
                    "date":    date_str,
                    "url":     url,
                    "content": content,
                })
                log.info(f"  ✔ [{date_str}] {title[:65]}")

            except Exception as exc:
                log.warning(f"  Error artikel ({exc.__class__.__name__}): {url[:60]}")

    log.info(f"  → {len(records)} artikel valid untuk keyword '{keyword}'.")
    return records

# ── 10. Main ──────────────────────────────────────────────────
def run() -> pd.DataFrame:
    log.info("=" * 58)
    log.info(" Detik PDP Scraper — MULAI")
    log.info(f" Rentang tahun : {YEAR_START}–{YEAR_END}")
    log.info(f" Total keyword : {len(KEYWORDS)}")
    log.info(f" Max halaman   : {MAX_PAGES} per keyword")
    log.info(f" Worker thread : {MAX_WORKERS}")
    log.info("=" * 58)

    all_records = []

    for kw in KEYWORDS:
        try:
            records = scrape_keyword(kw)
            all_records.extend(records)
        except Exception as exc:
            log.error(f"[Detik] Error fatal pada '{kw}': {exc}", exc_info=True)

    if not all_records:
        log.warning("[Detik] Tidak ada artikel yang terkumpul.")
        return pd.DataFrame()

    df = pd.DataFrame(all_records)

    # Deduplikasi URL (keyword berbeda bisa dapat artikel yang sama)
    before = len(df)
    df.drop_duplicates(subset=["url"], keep="first", inplace=True)
    log.info(f"\nDeduplikasi: {before} → {len(df)} baris.")

    # Filter ketat tahun (safety net terakhir)
    df = df[df["date"].apply(year_in_range)].copy()
    log.info(f"Setelah filter {YEAR_START}–{YEAR_END}: {len(df)} baris.")

    df = df[["source", "keyword", "title", "date", "url", "content"]]
    df.sort_values("date", ascending=False, inplace=True)
    df.reset_index(drop=True, inplace=True)

    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_ALL)
    log.info(f"✅ Disimpan ke '{OUTPUT_FILE}'")

    return df

# ── 11. Entry Point ───────────────────────────────────────────
if __name__ == "__main__":
    df = run()
    if not df.empty:
        print("\n── Ringkasan ────────────────────────────────────────")
        print(f"Total artikel  : {len(df)}")
        print(f"Rentang tanggal: {df['date'].min()} → {df['date'].max()}")
        print(f"\nPer keyword:")
        print(df["keyword"].value_counts().to_string())
        print(f"\nFile output    : {OUTPUT_FILE}")
        print("\n── 5 Artikel Terbaru ────────────────────────────────")
        print(df[["date", "title"]].head(5).to_string(index=False))


── Ringkasan ────────────────────────────────────────
Total artikel  : 5
Rentang tanggal: 2020-05-22 → 2020-12-07

Per keyword:
keyword
kebocoran data Dukcapil    2
data bocor                 1
hacker data                1
kebocoran data e-KTP       1

File output    : detik_pdp2020tok.csv

── 5 Artikel Terbaru ────────────────────────────────
      date                                                            title
2020-12-07               Fakta-fakta Gisel yang Ngaku Data Pribadinya Bocor
2020-06-20     Data Tes COVID-19 RI Disebut Bocor, Ini Tanggapan Menkominfo
2020-05-26 Aplikasi Berbahaya di Google Play Store, Mungkin Ada di Ponselmu
2020-05-22     Jutaan Data KPU Bocor, Pakar: Walau Terbuka Sangat Berbahaya
2020-05-22      Potensi Bahaya Besar Bayangi Kebocoran Data Penduduk di KPU


FIKS

In [ ]:
# ============================================================
# scraper_detik_fixed.py — Versi FIXED (filter tahun benar)
# Google Colab Ready | Standalone
# Output: detik_pdp.csv
# ============================================================
# !pip install requests beautifulsoup4 pandas lxml

import re
import csv
import time
import random
import logging
import requests
import pandas as pd
from bs4 import BeautifulSoup
from typing import Optional, Tuple

# ── 1. Logging ───────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ── 2. Konstanta — UBAH DI SINI ──────────────────────────────
KEYWORDS = [
    "perlindungan data pribadi",
    "UU PDP",
    "kebocoran data",
    "data pribadi bocor",
]

YEAR_START  = 2021   # ← ganti sesuai kebutuhan
YEAR_END    = 2021   # ← ganti sesuai kebutuhan
OUTPUT_FILE = "detik_pdp_2021_2.csv"
MAX_PAGES   = 1

# ── 3. HTTP Session ───────────────────────────────────────────
session = requests.Session()

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 "
    "(KHTML, like Gecko) Version/17.4 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:125.0) "
    "Gecko/20100101 Firefox/125.0",
]

def random_headers() -> dict:
    return {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Referer": "https://www.google.com/",
    }

def polite_sleep(lo: float = 1.0, hi: float = 3.0) -> None:
    time.sleep(random.uniform(lo, hi))

def safe_get(url: str, timeout: int = 12) -> Optional[BeautifulSoup]:
    try:
        resp = session.get(url, headers=random_headers(), timeout=timeout)
        if resp.status_code == 200:
            return BeautifulSoup(resp.text, "lxml")
        log.warning(f"HTTP {resp.status_code} → {url}")
    except requests.RequestException as exc:
        log.error(f"Request gagal ({exc.__class__.__name__}): {url}")
    return None

# ── 4. Parsing Tanggal ────────────────────────────────────────
def parse_meta_date(soup: BeautifulSoup) -> Optional[str]:
    """
    Ambil tanggal dari <meta name="publishdate" content="2024/03/15 10:30:00">.
    Ini adalah sumber tanggal PALING AKURAT untuk Detik.
    Format meta: YYYY/MM/DD HH:MM:SS → return YYYY-MM-DD
    """
    meta = soup.find("meta", attrs={"name": "publishdate"})
    if meta and meta.get("content"):
        m = re.match(r"(\d{4})/(\d{2})/(\d{2})", meta["content"].strip())
        if m:
            return f"{m.group(1)}-{m.group(2)}-{m.group(3)}"

    # Fallback 1: <meta property="article:published_time">
    meta2 = soup.find("meta", attrs={"property": "article:published_time"})
    if meta2 and meta2.get("content"):
        m = re.match(r"(\d{4})-(\d{2})-(\d{2})", meta2["content"].strip())
        if m:
            return f"{m.group(1)}-{m.group(2)}-{m.group(3)}"

    # Fallback 2: <time> tag dengan datetime attribute
    time_tag = soup.find("time", attrs={"itemprop": "datePublished"})
    if time_tag and time_tag.get("datetime"):
        m = re.match(r"(\d{4})-(\d{2})-(\d{2})", time_tag["datetime"])
        if m:
            return f"{m.group(1)}-{m.group(2)}-{m.group(3)}"

    return None

def year_in_range(date_str: Optional[str]) -> bool:
    """Cek apakah tahun dalam date string (YYYY-MM-DD) masuk rentang."""
    if not date_str:
        return False
    try:
        return YEAR_START <= int(date_str[:4]) <= YEAR_END
    except (ValueError, IndexError):
        return False

# ── 5. Pembersihan Teks ───────────────────────────────────────
_NOISE = re.compile(
    r"(Baca (Juga|juga)|BACA JUGA|Simak( juga| Video)?|"
    r"Advertisement|ADVERTISEMENT|Lihat Juga|Tonton Video|"
    r"Editor\s*:\s*[\w\s]+|Reporter\s*:\s*[\w\s]+|"
    r"ADVERTISEMENT SCROLL TO CONTINUE WITH CONTENT)",
    re.IGNORECASE,
)

def clean_text(raw: str) -> str:
    text = re.sub(r"<[^>]+>", " ", raw)
    text = _NOISE.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()

# ── 6. Deep Scrape Artikel ────────────────────────────────────
def extract_article(url: str) -> Tuple[Optional[str], str]:
    """
    Kunjungi halaman artikel, ambil:
      - tanggal dari <meta name="publishdate">   ← AKURAT
      - body content dari div.detail__body-text

    Mengembalikan (date_str, content_str).
    Suffix ?single=1 memaksa artikel multi-halaman jadi satu halaman.
    """
    fetch_url = url.rstrip("/") + "?single=1"
    soup = safe_get(fetch_url)
    if not soup:
        return None, ""

    # ── Tanggal dari meta (AKURAT) ────────────────────────────
    date_str = parse_meta_date(soup)

    # ── Body artikel ──────────────────────────────────────────
    body = soup.select_one("div.detail__body-text")
    if not body:
        body = soup.select_one("div.itp_bodycontent")   # fallback Detik lama
    if not body:
        body = soup.find("article")                      # fallback generik
    if not body:
        return date_str, ""

    for tag in body.find_all(["script", "style", "figure", "aside",
                               "noscript", "div.ads-box"]):
        tag.decompose()

    content = clean_text(body.get_text(separator=" "))
    return date_str, content

# ── 7. Build Search URL ───────────────────────────────────────
def build_search_url(keyword: str, page: int) -> str:
    """
    CATATAN: fromdate/todate di URL Detik tidak selalu dihormati server.
    Filter tahun yang reliable dilakukan via <meta name="publishdate">
    saat deep-scrape, bukan dari parameter URL ini.
    """
    return (
        "https://www.detik.com/search/searchall"
        f"?query={requests.utils.quote(keyword)}"
        f"&siteid=2&sortby=time&sorttime=0"
        f"&fromdate=01/01/{YEAR_START}&todate=31/12/{YEAR_END}"
        f"&page={page}"
    )

# ── 8. Scrape per Keyword ─────────────────────────────────────
def scrape_keyword(keyword: str) -> list[dict]:
    records = []
    stop_early = False   # flag: jika sudah masuk artikel terlalu lama (< YEAR_START)
    log.info(f"[Detik] Keyword: '{keyword}'")

    for page in range(1, MAX_PAGES + 1):
        if stop_early:
            break

        url = build_search_url(keyword, page)
        log.info(f"  → Halaman {page}: {url}")
        soup = safe_get(url)
        if not soup:
            break

        items = soup.select("article.list-content__item")
        if not items:
            log.info(f"  Tidak ada hasil di halaman {page}, berhenti.")
            break

        page_added = 0
        for item in items:
            try:
                title_tag = item.select_one("h3.media__title a")
                if not title_tag:
                    continue
                title = title_tag.get_text(strip=True)
                link  = title_tag.get("href", "")
                if not link:
                    continue

                polite_sleep(1.0, 2.5)

                # ── Deep scrape: tanggal AKURAT dari meta ────
                date_str, content = extract_article(link)

                if not date_str:
                    log.debug(f"  Tanggal tidak ditemukan, skip: {link}")
                    continue

                year = int(date_str[:4])

                # Artikel lebih baru dari batas atas → skip, lanjut
                if year > YEAR_END:
                    log.debug(f"  Skip (terlalu baru {date_str}): {title[:60]}")
                    continue

                # Artikel lebih tua dari batas bawah → semua halaman berikutnya
                # juga akan makin tua, bisa stop lebih awal
                if year < YEAR_START:
                    log.info(f"  Artikel sudah di luar rentang ({date_str}), "
                             f"stop pagination keyword ini.")
                    stop_early = True
                    break

                # Artikel dalam rentang YEAR_START–YEAR_END ✓
                if not content:
                    continue

                records.append({
                    "source":  "Detik.com",
                    "keyword": keyword,
                    "title":   title,
                    "date":    date_str,
                    "url":     link,
                    "content": content,
                })
                page_added += 1
                log.info(f"  ✔ [{date_str}] {title[:70]}")

            except Exception as exc:
                log.warning(f"  Item dilewati — {exc.__class__.__name__}: {exc}")
                continue

        log.info(f"  {page_added} artikel valid dari halaman {page}.")
        polite_sleep(2, 4)

    return records

# ── 9. Main ───────────────────────────────────────────────────
def run() -> pd.DataFrame:
    all_records = []

    for kw in KEYWORDS:
        try:
            records = scrape_keyword(kw)
            all_records.extend(records)
            log.info(f"[Detik] '{kw}' selesai → {len(records)} artikel valid.")
        except Exception as exc:
            log.error(f"[Detik] Error fatal pada '{kw}': {exc}", exc_info=True)
        polite_sleep(3, 6)

    if not all_records:
        log.warning("[Detik] Tidak ada artikel yang terkumpul.")
        return pd.DataFrame()

    df = pd.DataFrame(all_records)

    # Deduplikasi URL
    before = len(df)
    df.drop_duplicates(subset=["url"], keep="first", inplace=True)
    log.info(f"[Detik] Deduplikasi: {before} → {len(df)} baris.")

    # Filter final — double-check tahun (safety net)
    df = df[df["date"].apply(year_in_range)].copy()
    log.info(f"[Detik] Setelah filter ketat {YEAR_START}–{YEAR_END}: {len(df)} baris.")

    df = df[["source", "keyword", "title", "date", "url", "content"]]
    df.sort_values("date", ascending=False, inplace=True)
    df.reset_index(drop=True, inplace=True)

    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_ALL)
    log.info(f"✅ Disimpan ke '{OUTPUT_FILE}'")

    return df

# ── 10. Entry Point ───────────────────────────────────────────
if __name__ == "__main__":
    df = run()
    if not df.empty:
        print("\n── Ringkasan ────────────────────────────────────────")
        print(f"Total artikel  : {len(df)}")
        print(f"Rentang tanggal: {df['date'].min()} → {df['date'].max()}")
        print(f"Per keyword    :\n{df['keyword'].value_counts().to_string()}")
        print(f"\nFile output    : {OUTPUT_FILE}")
        print("\n── 5 Artikel Terbaru ────────────────────────────────")
        print(df[["date", "title"]].head(5).to_string(index=False))